# Sovereign Byte Firewall — Mamba-2 Accuracy A/B (Kaggle GPU)

Trains the **Mamba-2** backbone (`experiment/mamba-backbone`) on CIC-IDS2017 **Monday** (benign-only), so it can be compared against the transformer on the same held-out zero-day split. Throughput A/B is already done (see `MAMBA_EXPERIMENT.md`) — this notebook runs the still-pending **accuracy** A/B.

**Rules baked into this notebook:**
1. Train on the **Monday** file only (benign-only day). Never Wednesday (attack/eval day).
2. Same recipe as the transformer A/B: 20,000 steps, seq_len 512, batch 32 — for a fair comparison.
3. Smoke test first (20 steps) to prove the Mamba-2 wiring before the long run.
4. Checkpoint pushes to a **dedicated** HF Hub repo (`spst01/sovereign-byte-firewall-mamba2`) so it never mixes with the transformer's gs-numbered checkpoints.

Make sure GPU accelerator is ON: Settings > Accelerator > **GPU T4 x2** (Tesla P100 is NOT compatible — sm_60 < required sm_70).

## 1. Clone repo, checkout Mamba branch, merge latest main

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/TheSPST/sovereign-byte-firewall.git"
REPO_DIR = "/kaggle/working/sovereign-byte-firewall"
BRANCH = "experiment/mamba-backbone"

def run(cmd, **kw):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True, **kw)

if os.path.exists(REPO_DIR):
    run(["git", "-C", REPO_DIR, "fetch", "origin"])
else:
    run(["git", "clone", REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
# merge commits need an identity -- Kaggle containers have none set by default
run(["git", "config", "user.email", "kaggle-bot@users.noreply.github.com"])
run(["git", "config", "user.name", "kaggle-training-bot"])
run(["git", "checkout", BRANCH])
run(["git", "pull", "origin", BRANCH])

merge = subprocess.run(["git", "merge", "origin/main", "-m", "merge main into mamba branch"],
                        capture_output=True, text=True)
print(merge.stdout, merge.stderr)
if merge.returncode != 0:
    conflicted = subprocess.run(["git", "diff", "--name-only", "--diff-filter=U"],
                                 capture_output=True, text=True).stdout.split()
    print("Conflicted files:", conflicted)
    non_docs = [f for f in conflicted if not f.endswith(".md")]
    if non_docs:
        raise RuntimeError(
            f"Merge conflict in non-doc file(s) {non_docs} -- this touches real code, "
            f"not just docs, so it needs a human judgment call. Resolve manually inside "
            f"{REPO_DIR} (edit the conflict markers, git add, git commit), then re-run."
        )
    # Docs-only conflict (e.g. PROJECT_CONTEXT.md) -- safe to auto-resolve: keep this
    # branch's version and move on, since it doesn't affect training correctness.
    for f in conflicted:
        run(["git", "checkout", "--ours", f])
        run(["git", "add", f])
    run(["git", "commit", "--no-edit"])
    print("Docs-only merge conflict auto-resolved (kept mamba-branch version of):", conflicted)

print(subprocess.run(["git", "log", "--oneline", "-5"], capture_output=True, text=True).stdout)

## 2. Install deps (Mamba-2 needs the fused CUDA kernel + causal-conv1d)

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "scapy", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install",
                "causal-conv1d>=1.2.0", "mamba-ssm", "--no-build-isolation", "-q"], check=True)
print("deps installed")

## 3. GPU sanity check

In [ ]:
import torch, subprocess
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected! Turn on GPU accelerator in Settings.")
major, minor = torch.cuda.get_device_capability(0)
if major < 7:
    raise RuntimeError(f"GPU compute capability {major}.{minor} < 7.0 — switch accelerator to GPU T4 x2 (P100 is sm_60, unsupported).")
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"GPU[{i}]: {name} - {vram:.1f}GB VRAM")
subprocess.run(["nvidia-smi"])

## 4. Resolve Monday (benign-only) pcap — remote download if not already attached/cached

In [ ]:
import os
from huggingface_hub import hf_hub_download

MONDAY_HF_REPO = "bvsam/cic-ids-2017"
MONDAY_HF_FILENAME = "pcap/Monday-WorkingHours.pcap"
MONDAY_LOCAL_FALLBACK = "/kaggle/working/pcap/Monday-WorkingHours.pcap"
MONDAY_MIN_BYTES = 10 * 1e9  # sanity floor so a truncated download isn't reused

DATASET_PATH = None

# 1) Manually-attached Kaggle dataset (fastest — no remote download needed)
for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        if "onday" in f and f.endswith(".pcap"):
            DATASET_PATH = os.path.join(root, f)
            print("Monday pcap found (Kaggle input, no download needed):", DATASET_PATH)
            break
    if DATASET_PATH:
        break

# 2) Already downloaded to /kaggle/working from a prior run in this session
if not DATASET_PATH:
    if os.path.exists(MONDAY_LOCAL_FALLBACK) and os.path.getsize(MONDAY_LOCAL_FALLBACK) > MONDAY_MIN_BYTES:
        DATASET_PATH = MONDAY_LOCAL_FALLBACK
        print("Monday pcap already on local disk (skipping download):", DATASET_PATH)

# 3) Remote download from the public, non-gated HF dataset (~10 GB, one-time)
if not DATASET_PATH:
    print(f"Downloading {MONDAY_HF_FILENAME} from {MONDAY_HF_REPO} (~10 GB, one-time)...")
    DATASET_PATH = hf_hub_download(
        repo_id=MONDAY_HF_REPO,
        filename=MONDAY_HF_FILENAME,
        repo_type="dataset",
        local_dir="/kaggle/working/",
    )
    print("Downloaded:", DATASET_PATH)

base = os.path.basename(DATASET_PATH)
assert "onday" in base, "Resolved file is not the Monday benign capture!"
assert "ednesday" not in base, "Refusing to train on Wednesday (attack/eval day)!"
print(f"Training corpus: {DATASET_PATH} ({os.path.getsize(DATASET_PATH)/1e9:.2f} GB)")

## 5. Smoke test (proves the Mamba-2 wiring end-to-end, ~2 min, run before the long job)

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "train_ab.py",
                "--backbone", "mamba2", "--train_pcap", DATASET_PATH,
                "--steps", "20", "--seq_len", "128",
                "--out", "/kaggle/working/smoke_mamba2.pt"], check=True)
print("Smoke test passed — wiring confirmed end-to-end.")

## 6. Real Mamba-2 training run (same steps/seq_len as the transformer A/B, for a fair comparison)

In [ ]:
import os, sys, subprocess

TRAIN_OUT = "/kaggle/working/checkpoints/ckpt_mamba2.pt"
os.makedirs(os.path.dirname(TRAIN_OUT), exist_ok=True)

subprocess.run([sys.executable, "train_ab.py",
                "--backbone", "mamba2", "--train_pcap", DATASET_PATH,
                "--steps", "20000", "--seq_len", "512", "--batch_size", "32",
                "--out", TRAIN_OUT], check=True)

size_mb = os.path.getsize(TRAIN_OUT) / 1e6
print(f"Done. Checkpoint: {TRAIN_OUT} ({size_mb:.1f} MB)")

## 7. Push checkpoint to Hugging Face Hub (dedicated mamba2 repo — needs a Kaggle secret `HF_TOKEN`)

In [ ]:
import os
from huggingface_hub import HfApi, login
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN", "")

HF_CKPT_REPO = "spst01/sovereign-byte-firewall-mamba2"  # dedicated repo, kept apart from the transformer's

if not hf_token:
    print("WARNING: no HF_TOKEN secret found — skipping push.")
    print(f"Add a Kaggle secret named HF_TOKEN (write access) to push automatically. Checkpoint remains at: {TRAIN_OUT}")
else:
    login(token=hf_token, add_to_git_credential=False)
    api = HfApi()
    api.create_repo(repo_id=HF_CKPT_REPO, repo_type="model", exist_ok=True, private=False)
    remote_name = f"checkpoints/{os.path.basename(TRAIN_OUT)}"
    api.upload_file(path_or_fileobj=TRAIN_OUT, path_in_repo=remote_name,
                     repo_id=HF_CKPT_REPO, repo_type="model")
    print(f"Pushed to HF Hub: https://huggingface.co/{HF_CKPT_REPO}/blob/main/{remote_name}")
    print("Next: evaluate_zero_day.py --backbone mamba2 --checkpoint_path", TRAIN_OUT,
          "--score_agg topk --topk_frac 0.1  (compare AUC + detection@1%FPR vs the transformer)")